In [9]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

project_root = Path.cwd().parents[2]

# Make the src directory importable
sys.path.insert(0, str(project_root))

from src.data_loader import load_cv_data

# Load all five folds
fold_datasets = load_cv_data(project_root)

# Inspect the results
for fold_number, (train_df, val_df) in enumerate(
    fold_datasets, start=1
):
    print(
        f"Fold {fold_number}: "
        f"Training = {train_df.shape}, "
        f"Validation = {val_df.shape}"
    )

Fold 1: Training = (177324, 7), Validation = (42659, 7)
Fold 2: Training = (176273, 7), Validation = (43710, 7)
Fold 3: Training = (170186, 7), Validation = (49797, 7)
Fold 4: Training = (178311, 7), Validation = (41672, 7)
Fold 5: Training = (177838, 7), Validation = (42145, 7)


In [10]:

from src.feature_engineering import add_overall_history_features

v12_fold_datasets = []

for fold_number, (fold_train_df, fold_val_df) in enumerate(
    fold_datasets, start=1
):

    # Generate training features using past-only global prior
    v12_train_df = add_overall_history_features(
        fold_train_df,
        alpha=10,
        prior_data=fold_train_df
    )

    # Generate validation features using training data for global prior
    v12_val_df = add_overall_history_features(
        fold_val_df,
        alpha=10,
        prior_data=fold_train_df
    )

    v12_fold_datasets.append(
        (v12_train_df, v12_val_df)
    )

    print(
        f"Fold {fold_number}: "
        f"Training = {v12_train_df.shape}, "
        f"Validation = {v12_val_df.shape}"
    )


Fold 1: Training = (177324, 12), Validation = (42659, 12)
Fold 2: Training = (176273, 12), Validation = (43710, 12)
Fold 3: Training = (170186, 12), Validation = (49797, 12)
Fold 4: Training = (178311, 12), Validation = (41672, 12)
Fold 5: Training = (177838, 12), Validation = (42145, 12)


In [11]:
import numpy as np

fold_train_df, fold_val_df = v12_fold_datasets[0]

# Check the first few values
fold_train_df[
    ["previous_interactions", "log_previous_interactions"]
].head(10)

,previous_interactions,log_previous_interactions
0,0,0.000000
1,1,0.693147
2,2,1.098612
3,3,1.386294
4,4,1.609438
5,5,1.791759
6,6,1.945910
7,7,2.079442
8,8,2.197225
9,9,2.302585


In [12]:
expected = np.log1p(fold_train_df["previous_interactions"])

np.allclose(
    fold_train_df["log_previous_interactions"],
    expected
)

True

In [13]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, brier_score_loss

feature_columns = [
    "overall_smoothed_accuracy",
    "log_previous_interactions"
]

v12_models = []
v12_results = []

for fold_number, (train_df, val_df) in enumerate(v12_fold_datasets, start=1):

    X_train = train_df[feature_columns]
    y_train = train_df["correct"]

    X_val = val_df[feature_columns]
    y_val = val_df["correct"]

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(max_iter=1000))
    ])

    model.fit(X_train, y_train)

    y_pred = model.predict_proba(X_val)[:, 1]

    ll = log_loss(y_val, y_pred, labels=[0, 1])
    bs = brier_score_loss(y_val, y_pred)

    v12_models.append(model)

    v12_results.append({
        "fold": fold_number,
        "log_loss": ll,
        "brier_score": bs
    })

    print(
        f"Fold {fold_number}: "
        f"Log Loss = {ll:.4f}, "
        f"Brier Score = {bs:.4f}"
    )

Fold 1: Log Loss = 0.5944, Brier Score = 0.2035
Fold 2: Log Loss = 0.5900, Brier Score = 0.2013
Fold 3: Log Loss = 0.6042, Brier Score = 0.2080
Fold 4: Log Loss = 0.5947, Brier Score = 0.2036
Fold 5: Log Loss = 0.5860, Brier Score = 0.1997


In [14]:
for fold_number, model in enumerate(v12_models, start=1):
    coefficients = model.named_steps["classifier"].coef_[0]

    print(f"Fold {fold_number}")
    print(f"  Smoothed accuracy coefficient: {coefficients[0]:.4f}")
    print(f"  Log experience coefficient:    {coefficients[1]:.4f}")

Fold 1
  Smoothed accuracy coefficient: 0.6689
  Log experience coefficient:    0.0420
Fold 2
  Smoothed accuracy coefficient: 0.6861
  Log experience coefficient:    0.0379
Fold 3
  Smoothed accuracy coefficient: 0.6715
  Log experience coefficient:    0.0398
Fold 4
  Smoothed accuracy coefficient: 0.6801
  Log experience coefficient:    0.0390
Fold 5
  Smoothed accuracy coefficient: 0.6648
  Log experience coefficient:    0.0440


In [15]:
# Collect out-of-fold predictions for V1.2

v12_predictions = []

for fold_number, (train_data, val_data) in enumerate(
    v12_fold_datasets, start=1
):

    # Retrieve the already-trained model for this fold
    model = v12_models[fold_number - 1]

    # Generate validation probabilities
    X_val = val_data[feature_columns]
    y_pred_proba = model.predict_proba(X_val)[:, 1]

    # Store actual outcomes and predictions
    fold_predictions = pd.DataFrame({
        "user_id": val_data["user_id"].values,
        "order_id": val_data["order_id"].values,
        "fold": fold_number,
        "y_true": val_data["correct"].values,
        "y_pred_proba": y_pred_proba,
        "model": "V1.2"
    })

    v12_predictions.append(fold_predictions)

# Combine predictions from all five folds
v12_predictions_df = pd.concat(
    v12_predictions,
    ignore_index=True
)

print("Prediction collection completed.")
print("Total predictions:", len(v12_predictions_df))
display(v12_predictions_df.head())

Prediction collection completed.
Total predictions: 219983


,user_id,order_id,fold,y_true,y_pred_proba,model
0,21825,21616385,1,0,0.708463,V1.2
1,21825,21616478,1,1,0.640104,V1.2
2,21825,21616677,1,1,0.673569,V1.2
3,21825,21616919,1,1,0.700267,V1.2
4,21825,21671620,1,0,0.719518,V1.2


In [16]:
print("Total rows:", len(v12_predictions_df))

print("\nMissing values:")
print(v12_predictions_df.isna().sum())

print("\nFold distribution:")
print(v12_predictions_df["fold"].value_counts().sort_index())

print("\nProbability range:")
print(
    v12_predictions_df["y_pred_proba"].min(),
    v12_predictions_df["y_pred_proba"].max()
)

print("\nActual outcome distribution:")
print(v12_predictions_df["y_true"].value_counts())

Total rows: 219983

Missing values:
user_id         0
order_id        0
fold            0
y_true          0
y_pred_proba    0
model           0
dtype: int64

Fold distribution:
fold
1    42659
2    43710
3    49797
4    41672
5    42145
Name: count, dtype: int64

Probability range:
0.08150095678985494 0.9044568513130306

Actual outcome distribution:
y_true
1    144886
0     75097
Name: count, dtype: int64


In [17]:
prediction_dir = (
    project_root / "data" / "processed" / "model_predictions"
)

prediction_dir.mkdir(parents=True, exist_ok=True)

v12_prediction_path = prediction_dir / "v1_2_predictions.csv"

v12_predictions_df.to_csv(
    v12_prediction_path,
    index=False
)

print("V1.2 predictions saved successfully.")
print("File:", v12_prediction_path)

V1.2 predictions saved successfully.
File: c:\Users\rajur\OneDrive\Desktop\next-move-ai\data\processed\model_predictions\v1_2_predictions.csv


In [18]:
for name, value in list(globals().items()):
    if "fold" in name.lower() and isinstance(value, list):
        if len(value) > 0 and isinstance(value[0], tuple):
            train_df, val_df = value[0]

            print(f"\nVariable: {name}")
            print("Training columns:", train_df.columns.tolist())


Variable: fold_datasets
Training columns: ['user_id', 'order_id', 'problem_id', 'correct', 'skill_id', 'skill_set', 'attempt_count']

Variable: v12_fold_datasets
Training columns: ['user_id', 'order_id', 'problem_id', 'correct', 'skill_id', 'skill_set', 'attempt_count', 'previous_interactions', 'previous_correct', 'global_historical_prior', 'overall_smoothed_accuracy', 'log_previous_interactions']


In [19]:
feature_dir = (
    project_root
    / "data"
    / "processed"
    / "engineered_features"
    / "group_1_overall_history"
    / "v1.2"
)

feature_dir.mkdir(parents=True, exist_ok=True)

columns_to_save = [
    "user_id",
    "order_id",
    "correct",
    "overall_smoothed_accuracy",
    "log_previous_interactions"
]

for fold_number, (train_df, val_df) in enumerate(
    v12_fold_datasets, start=1
):
    train_df[columns_to_save].to_csv(
        feature_dir / f"fold_{fold_number}_train.csv",
        index=False
    )

    val_df[columns_to_save].to_csv(
        feature_dir / f"fold_{fold_number}_validation.csv",
        index=False
    )

    print(f"Fold {fold_number} saved.")

print("\nV1.2 features saved successfully!")
print("Location:", feature_dir)

Fold 1 saved.
Fold 2 saved.
Fold 3 saved.
Fold 4 saved.
Fold 5 saved.

V1.2 features saved successfully!
Location: c:\Users\rajur\OneDrive\Desktop\next-move-ai\data\processed\engineered_features\group_1_overall_history\v1.2


In [20]:
saved_df = pd.read_csv(feature_dir / "fold_1_train.csv")

print("Shape:", saved_df.shape)
print("Columns:", saved_df.columns.tolist())
print(saved_df.head())

Shape: (177324, 5)
Columns: ['user_id', 'order_id', 'correct', 'overall_smoothed_accuracy', 'log_previous_interactions']
   user_id  order_id  correct  overall_smoothed_accuracy  \
0       14  21617623        0                   0.733963   
1       14  21617632        1                   0.667158   
2       14  21617641        0                   0.694922   
3       14  21617650        0                   0.641398   
4       14  21617659        0                   0.595520   

   log_previous_interactions  
0                   0.000000  
1                   0.693147  
2                   1.098612  
3                   1.386294  
4                   1.609438  
